# Clinical NLP Pipeline - Brazilian Portuguese
# Pipeline de NLP Clinico - Portugues Brasileiro

This notebook demonstrates the complete clinical NLP pipeline for Brazilian Portuguese medical texts using IBM Watsonx Granite models.

Este notebook demonstra o pipeline completo de NLP clinico para textos medicos em Portugues Brasileiro utilizando modelos IBM Watsonx Granite.

## Pipeline Stages / Etapas do Pipeline

1. **Preprocessing** - Text normalization, abbreviation expansion, tokenization
2. **NER** - Clinical entity extraction (medications, dosages, conditions, procedures, lab values)
3. **Relation Extraction** - Medication-dosage, medication-condition relationships
4. **Summarization** - Structured clinical summary generation
5. **FHIR R4 Output** - Healthcare interoperability format

In [ ]:
import sys
import json
from pathlib import Path

# Add project root to path
project_root = str(Path.cwd().parent)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Project root: {project_root}")

## 1. Sample Clinical Texts / Textos Clinicos de Exemplo

Realistic synthetic clinical notes in Brazilian Portuguese.

In [ ]:
# Sample clinical texts (synthetic - no real patient data)
CLINICAL_TEXTS = {
    "consulta_cardiologia": (
        "Paciente com HAS em uso de Losartana 50mg 1x/dia e Anlodipino 5mg 1x/dia. "
        "DM2 controlada com Metformina 850mg 2x/dia. "
        "PA: 130x85 mmHg. FC: 72 bpm. "
        "Hemograma: HB 13,2 g/dL, PLT 245.000/mm3. "
        "Creatinina: 0,9 mg/dL. Glicemia de jejum: 112 mg/dL. "
        "HbA1c: 6,8%. "
        "ECG: ritmo sinusal, sem alteracoes. "
        "Conduta: manter medicacoes atuais, solicitar ecocardiograma."
    ),
    "internacao_pneumologia": (
        "Paciente internado com pneumonia comunitaria, em uso de Ceftriaxona 1g EV 1x/dia "
        "e Azitromicina 500mg VO 1x/dia. "
        "DPOC de base em uso de Formoterol + Budesonida inalatorio. "
        "SpO2: 93% em ar ambiente. FR: 22 irpm. "
        "Radiografia de torax com consolidacao em base direita. "
        "Hemograma: leucocitose com desvio a esquerda. PCR: 85 mg/dL."
    ),
    "evolucao_uti": (
        "Paciente no 3o dia de UTI pos IAM anterior extenso. "
        "Em uso de AAS 100mg, Clopidogrel 75mg, Atorvastatina 80mg, "
        "Enoxaparina 60mg SC 2x/dia, Metoprolol 50mg 2x/dia. "
        "Troponina: 15,2 ng/mL. CK-MB: 120 U/L. BNP: 850 pg/mL. "
        "ECO: FE 35%, hipocineisa anterior e septal. "
        "PA: 110x70 mmHg. FC: 68 bpm. "
        "Gasometria arterial: pH 7,38, pCO2 36, pO2 78, HCO3 22. "
        "Cateterismo cardiaco programado para amanha."
    ),
}

for name, text in CLINICAL_TEXTS.items():
    print(f"\n{'='*60}")
    print(f"Case: {name}")
    print(f"{'='*60}")
    print(text)

## 2. Preprocessing / Preprocessamento

### 2.1 Text Normalization

In [ ]:
from src.preprocessing import ClinicalTextNormalizer

normalizer = ClinicalTextNormalizer()

text = CLINICAL_TEXTS["consulta_cardiologia"]
normalized = normalizer.normalize(text)

print("Original text:")
print(text[:200], "...")
print(f"\nLength: {len(text)} chars")

print("\nNormalized text:")
print(normalized[:200], "...")
print(f"\nLength: {len(normalized)} chars")

### 2.2 Abbreviation Expansion

In [ ]:
from src.preprocessing import AbbreviationExpander

expander = AbbreviationExpander()

print(f"Total abbreviations loaded: {expander.abbreviation_count}")
print()

# Demonstrate abbreviation expansion
demo_text = "Paciente com HAS e DM2 em uso de IECA. PA 130x85 mmHg. ECG normal."

print("Original:")
print(demo_text)

print("\nExpanded (inline):")
print(expander.expand(demo_text, inline=True))

print("\nExpanded (parenthetical):")
print(expander.expand(demo_text, inline=False))

In [ ]:
# Look up specific abbreviations
abbreviations_to_check = ["HAS", "DM2", "IAM", "DPOC", "UTI", "IECA", "BRA", "SpO2"]

print("Abbreviation Lookups:")
print("-" * 60)
for abbrev in abbreviations_to_check:
    expansion = expander.lookup(abbrev)
    print(f"  {abbrev:8s} -> {expansion}")

### 2.3 Clinical Tokenization

In [ ]:
from src.preprocessing import ClinicalTokenizer

tokenizer = ClinicalTokenizer()

sample = "Losartana 50mg 1x/dia. PA: 130x85 mmHg. Hemoglobina: 12,5 g/dL."

tokens = tokenizer.tokenize(sample)

print(f"Text: {sample}")
print(f"\nTokens ({len(tokens)} total):")
print("-" * 50)
for t in tokens:
    print(f"  [{t.token_type:12s}] '{t.text}' ({t.start}:{t.end})")

In [ ]:
# Sentence splitting
text = CLINICAL_TEXTS["consulta_cardiologia"]
sentences = tokenizer.sentence_split(text)

print(f"Sentences ({len(sentences)} total):")
print("=" * 60)
for i, s in enumerate(sentences, 1):
    print(f"\n[{i}] {s.text}")
    print(f"    Tokens: {len(s.tokens)} | Span: {s.start}:{s.end}")

## 3. Named Entity Recognition (NER)

Extract clinical entities: medications, dosages, conditions, procedures, lab values.

In [ ]:
from src.ner import ClinicalNER

ner = ClinicalNER()

# Process first clinical text
text = CLINICAL_TEXTS["consulta_cardiologia"]
normalized = normalizer.normalize(text)

entities = ner.extract(normalized)

print(f"Entities extracted: {len(entities)}")
print("=" * 70)

for entity in entities:
    print(f"  [{entity.entity_type.value:25s}] '{entity.text}' "
          f"(conf: {entity.confidence:.2f}, src: {entity.source})")

In [ ]:
# Entity statistics
stats = ner.get_entity_stats(entities)

print("Entity Statistics:")
print(f"  Total entities: {stats['total']}")
print(f"  Average confidence: {stats['avg_confidence']:.3f}")
print("\n  By type:")
for entity_type, count in stats['by_type'].items():
    print(f"    {entity_type:25s}: {count}")

In [ ]:
# Process all clinical texts
print("NER Results for All Clinical Texts")
print("=" * 70)

all_results = {}

for name, text in CLINICAL_TEXTS.items():
    normalized = normalizer.normalize(text)
    entities = ner.extract(normalized)
    all_results[name] = entities
    
    stats = ner.get_entity_stats(entities)
    print(f"\n--- {name} ---")
    print(f"  Entities: {stats['total']} | Avg confidence: {stats['avg_confidence']:.3f}")
    for etype, count in stats['by_type'].items():
        if count > 0:
            print(f"    {etype}: {count}")

## 4. Relation Extraction

Identify relationships between clinical entities (medication-dosage, medication-condition, etc.).

In [ ]:
from src.ner import RelationExtractor

relation_extractor = RelationExtractor()

# Extract relations from first clinical text
text = CLINICAL_TEXTS["consulta_cardiologia"]
normalized = normalizer.normalize(text)
entities = ner.extract(normalized)
relations = relation_extractor.extract(entities, normalized)

print(f"Relations extracted: {len(relations)}")
print("=" * 80)

for rel in relations:
    print(f"  {rel.source.text} --[{rel.relation_type}]--> {rel.target.text} "
          f"(conf: {rel.confidence:.3f})")

In [ ]:
# Medication profile
profiles = relation_extractor.get_medication_profile(relations)

print("Medication Profile:")
print("=" * 60)

for profile in profiles:
    print(f"\n  Medication: {profile['medication']}")
    if profile['dosages']:
        print(f"  Dosages: {', '.join(profile['dosages'])}")
    if profile['conditions']:
        print(f"  Conditions: {', '.join(profile['conditions'])}")

## 5. Clinical Summarization

Generate structured clinical summaries (rule-based mode, no Watsonx API call required).

In [ ]:
from src.summarization import ClinicalSummarizer

summarizer = ClinicalSummarizer()

# Generate summary for first clinical text
text = CLINICAL_TEXTS["consulta_cardiologia"]
normalized = normalizer.normalize(text)
entities = ner.extract(normalized)
relations = relation_extractor.extract(entities, normalized)

summary = summarizer.summarize_from_entities(entities, relations, normalized)

print("Structured Clinical Summary:")
print("=" * 60)
print(json.dumps(summary, indent=2, ensure_ascii=False))

In [ ]:
# Summarize all clinical texts
print("Summaries for All Clinical Texts")
print("=" * 70)

for name, text in CLINICAL_TEXTS.items():
    normalized = normalizer.normalize(text)
    entities = ner.extract(normalized)
    relations = relation_extractor.extract(entities, normalized)
    summary = summarizer.summarize_from_entities(entities, relations, normalized)
    
    print(f"\n--- {name} ---")
    print(f"  Conditions: {summary['condicoes']}")
    print(f"  Medications: {len(summary['medicamentos'])} entries")
    for med in summary['medicamentos']:
        dosage = med.get('dosagem', 'N/A')
        print(f"    - {med['nome']} ({dosage})")
    print(f"  Procedures: {summary['procedimentos']}")
    print(f"  Lab Values: {summary['valores_laboratoriais']}")

## 6. FHIR R4 Output

Convert extracted entities and summaries to FHIR R4-compliant JSON for healthcare interoperability.

In [ ]:
from src.summarization import FHIRFormatter

fhir = FHIRFormatter()

# Generate FHIR Bundle from entities
text = CLINICAL_TEXTS["consulta_cardiologia"]
normalized = normalizer.normalize(text)
entities = ner.extract(normalized)

bundle = fhir.entities_to_bundle(entities, patient_id="synthetic-patient-001")

print(f"FHIR Bundle: {bundle['resourceType']} ({bundle['type']})")
print(f"Entries: {len(bundle['entry'])}")
print("=" * 60)

for entry in bundle['entry']:
    resource = entry['resource']
    rtype = resource['resourceType']
    if rtype == 'Condition':
        print(f"  [{rtype}] {resource['code']['text']}")
    elif rtype == 'MedicationStatement':
        print(f"  [{rtype}] {resource['medicationCodeableConcept']['text']}")
    elif rtype == 'Procedure':
        print(f"  [{rtype}] {resource['code']['text']}")
    elif rtype == 'Observation':
        print(f"  [{rtype}] {resource['valueString']}")

In [ ]:
# Show a single FHIR resource in detail
if bundle['entry']:
    print("Sample FHIR Resource (first entry):")
    print("=" * 60)
    print(json.dumps(bundle['entry'][0]['resource'], indent=2, ensure_ascii=False))

In [ ]:
# Generate FHIR Composition from summary
text = CLINICAL_TEXTS["consulta_cardiologia"]
normalized = normalizer.normalize(text)
entities = ner.extract(normalized)
relations = relation_extractor.extract(entities, normalized)
summary = summarizer.summarize_from_entities(entities, relations, normalized)

composition = fhir.summary_to_composition(summary, patient_id="synthetic-patient-001")

print("FHIR Composition:")
print("=" * 60)
print(f"  Title: {composition['title']}")
print(f"  Status: {composition['status']}")
print(f"  Sections: {len(composition['section'])}")
print()

for section in composition['section']:
    print(f"  Section: {section['title']}")
    print(f"    Content: {section['text']['div'][:100]}...")

## 7. End-to-End Pipeline / Pipeline Completo

Complete processing of a clinical note through all pipeline stages.

In [ ]:
def process_clinical_text(raw_text: str) -> dict:
    """Run the full clinical NLP pipeline on a text."""
    # Step 1: Normalize
    normalized = normalizer.normalize(raw_text)
    
    # Step 2: Expand abbreviations
    expanded = expander.expand(normalized, inline=False)
    
    # Step 3: Extract entities
    entities = ner.extract(normalized)
    
    # Step 4: Extract relations
    relations = relation_extractor.extract(entities, normalized)
    
    # Step 5: Generate summary
    summary = summarizer.summarize_from_entities(entities, relations, normalized)
    
    # Step 6: Generate FHIR output
    bundle = fhir.entities_to_bundle(entities)
    composition = fhir.summary_to_composition(summary)
    
    return {
        "normalized_text": normalized,
        "expanded_text": expanded,
        "entities": [e.to_dict() for e in entities],
        "relations": [r.to_dict() for r in relations],
        "summary": summary,
        "fhir_bundle": bundle,
        "fhir_composition": composition,
    }

# Process the UTI evolution note
result = process_clinical_text(CLINICAL_TEXTS["evolucao_uti"])

print("End-to-End Pipeline Result (evolucao_uti)")
print("=" * 60)
print(f"  Entities: {len(result['entities'])}")
print(f"  Relations: {len(result['relations'])}")
print(f"  FHIR Bundle entries: {len(result['fhir_bundle']['entry'])}")
print(f"  Composition sections: {len(result['fhir_composition']['section'])}")
print()
print("Summary:")
print(json.dumps(result['summary'], indent=2, ensure_ascii=False))

---

## Author / Autor

**Gabriel Demetrios Lafis**
- GitHub: [galafis](https://github.com/galafis)
- LinkedIn: [gabriel-demetrios-lafis](https://www.linkedin.com/in/gabriel-demetrios-lafis/)